# Data Preparation

This notebook turns the raw `dataset_malwares.csv` into a cleaned copy: rows with missing feature values are dropped, and rows that exactly duplicate another row's 77 feature values are dropped (a duplicate landing in both a training and test split would let a model memorise that row instead of genuinely generalising, as found in `01_data_understanding.ipynb`).

Each cleaning step below is its own cell, so the effect of every step is visible on its own rather than hidden inside a function. `04_modelling_classical.ipynb`, `05_modelling_mlp.ipynb`, and `06_evaluation.ipynb` repeat these same steps inline, scoped to whichever feature set they train on (`ORDER_OF_FEATURES`, `CORE_TRAITS`, `BEHAVIORAL_FEATURES`, or `RAW_HEADER_FEATURES`, see `constants.py`), rather than all depending on one saved file here. This notebook applies the steps once with all 77 features, to produce a general cleaned dataset for `03_eda.ipynb` to explore visually.

In [1]:
# Standard library
import sys

# Third-party
import pandas as pd

pd.set_option("display.max_columns", None)
sys.path.append("../src")

from constants import ORDER_OF_FEATURES, ID_COLUMNS, LABEL_COLUMN

## Load the Raw Dataset

In [2]:
# Same raw CSV used in 01_data_understanding.ipynb
df = pd.read_csv("../data/dataset_malwares.csv")
print(df.shape)

(19611, 79)


19,611 rows and 79 columns, matching `01_data_understanding.ipynb`.

## Split into Features, Label, and Name

Separated up front so every cleaning step below only has to look at the 77 feature columns, `X`, not accidentally touch `Name` (an identifier, `ID_COLUMNS`) or `Malware` (the label, `LABEL_COLUMN`).

In [3]:
X = df[ORDER_OF_FEATURES].copy()
y = df[LABEL_COLUMN].copy()
names = df[ID_COLUMNS].copy()
print(f"X: {X.shape}, y: {y.shape}, names: {names.shape}")

X: (19611, 77), y: (19611,), names: (19611, 1)


## Drop Rows with Missing Feature Values

`01_data_understanding.ipynb` already found 0 missing values in this dataset, but the check runs here anyway, defensively, in case a future dataset update introduces gaps this one does not have.

In [4]:
before = len(X)
mask = X.notna().all(axis=1)
X, y, names = X[mask], y[mask], names[mask]
dropped_na = before - len(X)
print(f"Dropped {dropped_na} rows with missing values. {len(X)} rows remain.")

Dropped 0 rows with missing values. 19611 rows remain.


## Drop Duplicate Feature Rows

`01_data_understanding.ipynb` found 2,984 rows that share identical values across all 77 features with at least one other row (only `Name` differs). If left in, the exact same feature vector could land in both the training and test split, letting a model memorise that row instead of genuinely generalising.

In [5]:
before = len(X)
dup_mask = ~X.duplicated()
X, y, names = X[dup_mask], y[dup_mask], names[dup_mask]
dropped_dup = before - len(X)
print(f"Dropped {dropped_dup} duplicate feature rows. {len(X)} rows remain.")

Dropped 2984 duplicate feature rows. 16627 rows remain.


16,627 rows remain after dropping the 2,984 duplicate-feature rows, exactly matching the count `01_data_understanding.ipynb` found (19,611 - 2,984 = 16,627, 0 rows lost to missing values).

## Check the Class Balance After Cleaning

In [6]:
y.value_counts(normalize=True) * 100

Malware
1    70.156974
0    29.843026
Name: proportion, dtype: float64

The split shifted from 74.44% malicious / 25.56% benign (raw) to 70.16% malicious / 29.84% benign (cleaned): removing the 2,984 duplicate-feature rows removed proportionally more malicious rows than benign ones (likely the same malware sample submitted to VirusShare more than once), rebalancing the dataset slightly toward benign. It is still a substantial malicious majority, so every model trained later still needs `class_weight="balanced"` (classical models) or a computed `scale_pos_weight` (XGBoost), not a smaller fix.

## Assemble and Save the Cleaned Dataset

The cleaned dataset keeps `Name`, all 77 features, and `Malware`, in the same column order as the raw file, so it remains a drop-in replacement wherever the raw file was used.

In [7]:
cleaned_df = pd.concat([names, X, y], axis=1)
cleaned_df.to_csv("../data/dataset_cleaned.csv", index=False)
cleaned_df.shape

(16627, 79)

In [8]:
# Preview to confirm the saved file has the expected shape and columns
cleaned_df.head()

,Name,e_magic,e_cblp,e_cp,e_crlc,e_cparhdr,e_minalloc,e_maxalloc,e_ss,e_sp,e_csum,e_ip,e_cs,e_lfarlc,e_ovno,e_oemid,e_oeminfo,e_lfanew,Machine,NumberOfSections,TimeDateStamp,PointerToSymbolTable,NumberOfSymbols,SizeOfOptionalHeader,Characteristics,Magic,MajorLinkerVersion,MinorLinkerVersion,SizeOfCode,SizeOfInitializedData,SizeOfUninitializedData,AddressOfEntryPoint,BaseOfCode,ImageBase,SectionAlignment,FileAlignment,MajorOperatingSystemVersion,MinorOperatingSystemVersion,MajorImageVersion,MinorImageVersion,MajorSubsystemVersion,MinorSubsystemVersion,SizeOfHeaders,CheckSum,SizeOfImage,Subsystem,DllCharacteristics,SizeOfStackReserve,SizeOfStackCommit,SizeOfHeapReserve,SizeOfHeapCommit,LoaderFlags,NumberOfRvaAndSizes,SuspiciousImportFunctions,SuspiciousNameSection,SectionsLength,SectionMinEntropy,SectionMaxEntropy,SectionMinRawsize,SectionMaxRawsize,SectionMinVirtualsize,SectionMaxVirtualsize,SectionMaxPhysical,SectionMinPhysical,SectionMaxVirtual,SectionMinVirtual,SectionMaxPointerData,SectionMinPointerData,SectionMaxChar,SectionMainChar,DirectoryEntryImport,DirectoryEntryImportSize,DirectoryEntryExport,ImageDirectoryEntryExport,ImageDirectoryEntryImport,ImageDirectoryEntryResource,ImageDirectoryEntryException,ImageDirectoryEntrySecurity,Malware
0,VirusShare_a878ba26000edaac5c98eff4432723b3,23117,144,3,0,4,0,65535,0,184,0,0,0,64,0,0,0,248,34404,6,1236512358,0,0,240,34,523,8,0,54784,189440,0,51316,4096,4294967296,4096,512,6,0,6,0,5,2,1024,295281,274432,2,32832,524288,8192,1048576,4096,0,16,0,0,6,0.000000,0,512,0,274,0,188416,0,270336,0,245248,0,3758096608,0,7,152,0,0,54440,77824,73728,0,1
1,VirusShare_ef9130570fddc174b312b2047f5f4cf0,23117,144,3,0,4,0,65535,0,184,0,0,0,64,0,0,0,240,332,5,1365109591,0,0,224,258,267,9,0,205824,139264,0,84654,4096,4194304,4096,512,5,0,0,0,5,0,1024,0,442368,2,33088,1048576,4096,1048576,4096,0,16,0,0,5,3.815281,0,8704,0,24124,0,205680,0,339968,0,314880,0,3791650880,0,16,311,0,0,262276,294912,0,346112,1
2,VirusShare_ef84cdeba22be72a69b198213dada81a,23117,144,3,0,4,0,65535,0,184,0,0,0,64,0,0,0,256,332,6,1438777028,0,0,224,14,267,6,0,24576,20480,0,27364,256,4194304,4096,4096,4,0,0,0,4,0,4096,0,49152,2,0,1048576,4096,1048576,69632,0,528,0,0,6,0.103538,0,4096,0,329,0,24065,0,45056,0,45056,0,3221225536,0,6,176,0,0,36864,40960,0,0,1
3,VirusShare_6bf3608e60ebc16cbcff6ed5467d469e,23117,144,3,0,4,0,65535,0,184,0,0,0,64,0,0,0,128,332,7,1354629311,0,0,224,783,267,2,22,34304,28160,297472,16685,4096,4194304,4096,512,4,0,6,0,4,0,1024,14174816,1032192,2,32768,2097152,4096,1048576,4096,0,16,14,0,7,0.000000,0,0,0,144,0,638976,0,1003520,0,58880,0,3224371328,0,8,155,0,0,356352,1003520,0,14109472,1
4,VirusShare_2cc94d952b2efb13c7d6bbe0dd59d3fb,23117,144,3,0,4,0,65535,0,184,0,0,0,64,0,0,0,128,332,7,1386631250,0,0,224,783,267,2,56,8192,89600,512,4416,4096,4194304,4096,512,4,0,1,0,4,0,1024,0,110592,2,0,2097152,4096,1048576,4096,0,16,2,0,7,0.000000,0,0,0,24,0,42916,0,73728,0,54784,0,3227516992,0,2,43,0,0,61440,73728,0,90624,1


## Summary

- Loaded the raw `dataset_malwares.csv` (19,611 rows, 79 columns) and split it into `X` (77 features), `y` (the `Malware` label), and `names` (`Name`), so each cleaning step below only touches the feature columns.
- Dropped rows with missing feature values as its own step: 0 rows dropped, confirming `01_data_understanding.ipynb`'s finding.
- Dropped duplicate-feature rows as its own step: 2,984 rows dropped, 16,627 remain.
- Confirmed the class balance shifted from 74.44%/25.56% (raw) to 70.16%/29.84% (cleaned) malicious/benign, dropping duplicates removed proportionally more malicious rows, but the dataset remains malicious-majority.
- Saved the cleaned dataset to `data/dataset_cleaned.csv` for `03_eda.ipynb` to explore visually.
- `04_modelling_classical.ipynb`, `05_modelling_mlp.ipynb`, and `06_evaluation.ipynb` each repeat these same steps inline against the raw CSV, scoped to their own feature set (`CORE_TRAITS` for the recommended model, `ORDER_OF_FEATURES` for the full feature-set comparison), rather than depending on this notebook's saved file.